# Ollama + Qwen3:8b — OpenAI SDK Tests

**Ollama endpoint:** `http://192.168.0.80:31436`  
**Model:** `qwen3:8b`

Sections:
1. Connectivity check
2. Structured outputs (Pydantic)
3. Conversation history
4. Tool calling

## 1. Setup & Connectivity Check

In [1]:
from openai import OpenAI

OLLAMA_BASE_URL = "http://192.168.0.80:31436/v1"
MODEL = "qwen3:8b"

client = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

# Quick connectivity check
models = client.models.list()
print("Available models:")
for m in models.data:
    print(f"  - {m.id}")

assert any(m.id == MODEL for m in models.data), f"{MODEL} not found!"
print(f"\n✅ {MODEL} is available")

Available models:
  - qwen3:8b

✅ qwen3:8b is available


In [3]:
# Simple completion test
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Say hello in 10 words or less."}],
)

print("Response:", response.choices[0].message.content)
print(f"Tokens — prompt: {response.usage.prompt_tokens}, completion: {response.usage.completion_tokens}")

Response: Hello! 😊 How can I assist you?
Tokens — prompt: 20, completion: 265


## 2. Structured Outputs with Pydantic

In [5]:
import json
from pydantic import BaseModel


class Country(BaseModel):
    name: str
    capital: str
    population_millions: float
    continent: str


class CountryList(BaseModel):
    countries: list[Country]


# Ask the model to return structured JSON matching the schema
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "system",
            "content": (
                "You are a helpful assistant. Always respond with valid JSON only, no markdown, no extra text. "
                f"Use this exact JSON schema: {CountryList.model_json_schema()}"
            ),
        },
        {"role": "user", "content": "List 3 countries in Asia with their capitals and approximate population."},
    ],
    temperature=0.2,
)

raw = response.choices[0].message.content
print("Raw response:\n", raw)

# Parse & validate with Pydantic
parsed = CountryList.model_validate_json(raw)
print("\n✅ Parsed Pydantic object:")
for c in parsed.countries:
    print(f"  {c.name} — capital: {c.capital}, pop: {c.population_millions}M, continent: {c.continent}")

Raw response:
 {"countries": [{"name": "China", "capital": "Beijing", "population_millions": 1400, "continent": "Asia"}, {"name": "India", "capital": "New Delhi", "population_millions": 1380, "continent": "Asia"}, {"name": "Japan", "capital": "Tokyo", "population_millions": 126, "continent": "Asia"}]}

✅ Parsed Pydantic object:
  China — capital: Beijing, pop: 1400.0M, continent: Asia
  India — capital: New Delhi, pop: 1380.0M, continent: Asia
  Japan — capital: Tokyo, pop: 126.0M, continent: Asia


In [8]:
from agents import Agent, OpenAIChatCompletionsModel, Runner
from openai import AsyncOpenAI
from pydantic import BaseModel, Field
import asyncio


OLLAMA_BASE_URL = "http://192.168.0.80:31436/v1"
MODEL = "qwen3:8b"


groq_client = AsyncOpenAI(
    api_key="groq",  # Note: This looks like it's for Ollama, not Groq
    base_url=OLLAMA_BASE_URL,
)

# --- Schema ---
class Country(BaseModel):
    name: str = Field(description="Country name")
    capital: str = Field(description="Capital city")
    population_millions: float = Field(description="Population in millions")
    continent: str = Field(description="Continent name")

class CountryList(BaseModel):
    countries: list[Country]

# --- Agent ---
writer_agent = Agent(
    name="CountryAgent",
    instructions=(
        "You are a helpful assistant that provides country information."
    ),
    model=OpenAIChatCompletionsModel(
        model=MODEL, 
        openai_client=groq_client,
    ),
    output_type=CountryList,
)

# --- Run in notebook ---
async def main():
    result = await Runner.run(
        writer_agent,
        input="List 3 countries in Asia with their capitals and approximate population.",
    )
    parsed: CountryList = result.final_output
    print("✅ Parsed successfully!")
    for c in parsed.countries:
        print(f"  {c.name} — capital: {c.capital}, pop: {c.population_millions}M, continent: {c.continent}")

# For Jupyter notebook - use await directly
await main()

✅ Parsed successfully!
  China — capital: Beijing, pop: 1400.0M, continent: Asia
  India — capital: New Delhi, pop: 1380.0M, continent: Asia
  Indonesia — capital: Jakarta, pop: 273.0M, continent: Asia


In [13]:
# Structured output: nested model
from typing import Optional


class Address(BaseModel):
    street: str
    city: str
    zip_code: str


class Person(BaseModel):
    name: str
    age: int
    occupation: str
    address: Address
    hobbies: list[str]


response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "system",
            "content": (
                "Respond with valid JSON only, no markdown fences, no extra text. "
                f"Schema: {Person.model_json_schema()}"
            ),
        },
        {"role": "user", "content": "Invent a fictional person living in Tokyo."},
    ],
    temperature=0.7,
)

raw = response.choices[0].message.content
print("Raw:\n", raw)
person = Person.model_validate_json(raw)
print(f"\n✅ {person.name}, age {person.age}, lives at {person.address.city}")
print(f"   Hobbies: {', '.join(person.hobbies)}")

Raw:
 {
  "name": "Haruki Tanaka",
  "age": 28,
  "occupation": "Software Engineer",
  "address": {
    "street": "1-8-1 Nishishinjuku",
    "city": "Tokyo",
    "zip_code": "160-8011"
  },
  "hobbies": ["anime collecting", "mountain hiking", "traditional tea ceremony", "photography"]
}

✅ Haruki Tanaka, age 28, lives at Tokyo
   Hobbies: anime collecting, mountain hiking, traditional tea ceremony, photography


In [14]:
from agents import Agent, OpenAIChatCompletionsModel, Runner
from openai import AsyncOpenAI
from pydantic import BaseModel, Field

OLLAMA_BASE_URL = "http://192.168.0.80:31436/v1"
MODEL = "qwen3:8b"

ollama_client = AsyncOpenAI(
    api_key="ollama",
    base_url=OLLAMA_BASE_URL,
)

# --- Schema ---
class Address(BaseModel):
    street: str
    city: str
    zip_code: str

class Person(BaseModel):
    name: str
    age: int
    occupation: str
    address: Address
    hobbies: list[str]

# --- Agent ---
person_agent = Agent(
    name="PersonAgent",
    instructions="You are a creative assistant that invents fictional people.",
    model=OpenAIChatCompletionsModel(
        model=MODEL,
        openai_client=ollama_client,
    ),
    output_type=Person,
)

# --- Run ---
result = await Runner.run(
    person_agent,
    input="Invent a fictional person living in Tokyo.",
)

person: Person = result.final_output
print(f"✅ {person.name}, age {person.age}, lives at {person.address.city}")
print(f"   Occupation: {person.occupation}")
print(f"   Hobbies: {', '.join(person.hobbies)}")
print(f"   Address: {person.address.street}, {person.address.zip_code}")

✅ Haruka Sato, age 28, lives at Tokyo
   Occupation: Digital Calligrapher & AI Art Curator
   Hobbies: Stargazing in Roppongi Hills, Collecting vintage teacups, Writing haikus for AI-generated landscapes, Teaching coding to underprivileged kids via a nonprofit called 'Code & Calligraphy'
   Address: Shibuya-dori 5-7-9, 108-0011


## 3. Conversation History

In [9]:
# Multi-turn conversation — the model should remember prior context
history = [
    {"role": "system", "content": "You are a concise assistant. Keep replies under 50 words."},
]


def chat(user_msg: str) -> str:
    """Send a message, append to history, return assistant reply."""
    history.append({"role": "user", "content": user_msg})
    resp = client.chat.completions.create(model=MODEL, messages=history)
    reply = resp.choices[0].message.content
    history.append({"role": "assistant", "content": reply})
    return reply


print("User: My name is Alice.")
print("AI:  ", chat("My name is Alice."))

print("\nUser: What is the capital of France?")
print("AI:  ", chat("What is the capital of France?"))

print("\nUser: What is my name?")
print("AI:  ", chat("What is my name?"))

print("\n--- Full history ---")
for msg in history:
    print(f"  [{msg['role']}] {msg['content'][:120]}")

User: My name is Alice.
AI:   Hello, Alice! How can I assist you today?

User: What is the capital of France?
AI:   The capital of France is Paris.

User: What is my name?
AI:   Your name is Alice.

--- Full history ---
  [system] You are a concise assistant. Keep replies under 50 words.
  [user] My name is Alice.
  [assistant] Hello, Alice! How can I assist you today?
  [user] What is the capital of France?
  [assistant] The capital of France is Paris.
  [user] What is my name?
  [assistant] Your name is Alice.


## 4. Tool Calling (Function Calling)

In [10]:
import json

# Define tools the model can call
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a given city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "City name"},
                    "unit": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "description": "Temperature unit",
                    },
                },
                "required": ["city"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Evaluate a mathematical expression and return the result.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {"type": "string", "description": "Math expression, e.g. '2 + 2'"},
                },
                "required": ["expression"],
            },
        },
    },
]


# Mock implementations
def get_weather(city: str, unit: str = "celsius") -> str:
    # Fake weather data
    data = {"city": city, "temp": 22, "unit": unit, "condition": "Partly cloudy"}
    return json.dumps(data)


def calculate(expression: str) -> str:
    # Safe math eval — only digits and operators
    allowed = set("0123456789+-*/.() ")
    if not all(ch in allowed for ch in expression):
        return json.dumps({"error": "Invalid expression"})
    try:
        result = eval(expression)  # safe: restricted to numeric chars
        return json.dumps({"result": result})
    except Exception as e:
        return json.dumps({"error": str(e)})


TOOL_MAP = {"get_weather": get_weather, "calculate": calculate}

In [11]:
def run_with_tools(user_msg: str) -> str:
    """Send a message, handle tool calls if any, return final answer."""
    messages = [
        {"role": "system", "content": "You are a helpful assistant. Use the provided tools when needed."},
        {"role": "user", "content": user_msg},
    ]

    response = client.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    msg = response.choices[0].message

    # If no tool calls, return the content directly
    if not msg.tool_calls:
        print("  (No tool calls made)")
        return msg.content

    # Process each tool call
    messages.append(msg)
    for tc in msg.tool_calls:
        fn_name = tc.function.name
        fn_args = json.loads(tc.function.arguments)
        print(f"  🔧 Tool call: {fn_name}({fn_args})")

        result = TOOL_MAP[fn_name](**fn_args)
        print(f"  📦 Result: {result}")

        messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})

    # Get final response after tool results
    final = client.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    return final.choices[0].message.content


# Test 1: Weather
print("=== Weather Query ===")
print("User: What's the weather in Tokyo?")
answer = run_with_tools("What's the weather in Tokyo?")
print(f"AI: {answer}")

print("\n=== Math Query ===")
print("User: What is 123 * 456 + 789?")
answer = run_with_tools("What is 123 * 456 + 789?")
print(f"AI: {answer}")

print("\n=== No-Tool Query ===")
print("User: Tell me a short joke.")
answer = run_with_tools("Tell me a short joke.")
print(f"AI: {answer}")

=== Weather Query ===
User: What's the weather in Tokyo?
  🔧 Tool call: get_weather({'city': 'Tokyo'})
  📦 Result: {"city": "Tokyo", "temp": 22, "unit": "celsius", "condition": "Partly cloudy"}
AI: The current weather in Tokyo is partly cloudy with a temperature of 22°C.

=== Math Query ===
User: What is 123 * 456 + 789?
  🔧 Tool call: calculate({'expression': '123 * 456 + 789'})
  📦 Result: {"result": 56877}
AI: The result of $123 \times 456 + 789$ is **56,877**.

=== No-Tool Query ===
User: Tell me a short joke.
  (No tool calls made)
AI: Here's a short joke for you:

Why did the chicken cross the road?  
To get to the other side! 🐔


## 5. Tool Calling + Conversation History Combined

In [12]:
class ToolChat:
    """Chat session with persistent history and tool support."""

    def __init__(self):
        self.messages = [
            {"role": "system", "content": "You are a helpful assistant. Use tools when appropriate. Be concise."},
        ]

    def send(self, user_msg: str) -> str:
        self.messages.append({"role": "user", "content": user_msg})
        response = client.chat.completions.create(model=MODEL, messages=self.messages, tools=tools)
        msg = response.choices[0].message

        if not msg.tool_calls:
            self.messages.append({"role": "assistant", "content": msg.content})
            return msg.content

        # Handle tool calls
        self.messages.append(msg)
        for tc in msg.tool_calls:
            fn_name = tc.function.name
            fn_args = json.loads(tc.function.arguments)
            print(f"  🔧 {fn_name}({fn_args})")
            result = TOOL_MAP[fn_name](**fn_args)
            self.messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})

        final = client.chat.completions.create(model=MODEL, messages=self.messages, tools=tools)
        reply = final.choices[0].message.content
        self.messages.append({"role": "assistant", "content": reply})
        return reply


session = ToolChat()

queries = [
    "What's the weather in London?",
    "How about in Paris?",
    "Now compute 42 * 58.",
    "Which city from our conversation had what weather?",
]

for q in queries:
    print(f"\nUser: {q}")
    print(f"AI:   {session.send(q)}")


User: What's the weather in London?
  🔧 get_weather({'city': 'London'})
AI:   The current weather in London is **22°C** with "Partly cloudy" conditions. ☁️ Let me know if you'd like more details!

User: How about in Paris?
  🔧 get_weather({'city': 'Paris', 'unit': 'celsius'})
AI:   The current weather in Paris is **22°C** with "Partly cloudy" conditions. ☁️ Let me know if you'd like more details!

User: Now compute 42 * 58.
  🔧 calculate({'expression': '42 * 58'})
AI:   The result of $42 \times 58$ is **2436**. 

Let me know if you need help with anything else! 😊

User: Which city from our conversation had what weather?
AI:   London and Paris both have **22°C** with "Partly cloudy" conditions. ☁️ Let me know if you need further details!


In [ ]:
print("\n✅ All tests passed — Ollama qwen3:8b works with OpenAI SDK for:")
print("   • Basic chat completions")
print("   • Structured outputs (Pydantic)")
print("   • Multi-turn conversation history")
print("   • Tool / function calling")